# Limpieza, análisis y estructuración de datos con Pandas (Instacart)
## Lección 2: Limpieza y estandarización (Laboratorio guiado)

In [1]:
import pandas as pd
from pathlib import Path

# Sistema robusto para encontrar la ruta de los datos automáticamente
if Path('../data/raw').exists():
    RUTA = Path('../data/raw')
    SALIDA = Path('../data/processed')
elif Path('data/raw').exists():
    RUTA = Path('data/raw')
    SALIDA = Path('data/processed')
else:
    raices = (Path.cwd(), *Path.cwd().parents)
    raiz = next((r for r in raices if (r / 'data' / 'raw').is_dir()), Path.cwd())
    RUTA = raiz / 'data' / 'raw'
    SALIDA = raiz / 'data' / 'processed'

SALIDA.mkdir(parents=True, exist_ok=True)

# 1. Cargar datos crudos
orders_crudo = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo = pd.read_csv(RUTA / 'order_products.csv', sep=';')

# 2. Función de registro
registro = []
def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla, 'paso': paso, 'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 2)
    })
    return df

# 3. Funciones de limpieza
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8', 'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True).str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

# 4. Ejecutar Pipeline
orders_limpio = (orders_crudo.pipe(anotar, 'orders', '0 crudo')
                 .pipe(quitar_duplicados).pipe(anotar, 'orders', '1 sin duplicados')
                 .pipe(tipos_orders).pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo.pipe(anotar, 'products', '0 crudo')
                   .pipe(limpiar_nombres).pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo.pipe(anotar, 'order_prods', '0 crudo')
             .pipe(tipos_order_prods).pipe(anotar, 'order_prods', '1 tipos'))

# 5. Mostrar informe
display(pd.DataFrame(registro))

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,22.99
1,orders,1 sin duplicados,478952,28817,22.99
2,orders,2 tipos,478952,28817,7.18
3,products,0 crudo,49694,1258,5.11
4,products,1 nombres,49694,0,5.14
5,order_prods,0 crudo,4545007,836,145.44
6,order_prods,1 tipos,4545007,836,54.54


In [2]:
# Reto 2: Resolviendo las 7 preguntas de la guía
print("--- 1. Duplicados en order_prods ---")
print("Exactos:", op_limpio.duplicated().sum())
print("Pares repetidos:", op_limpio.duplicated(subset=['order_id', 'product_id']).sum())
print("Valores en reordered:", op_limpio['reordered'].unique())

print("\n--- 2. Pasillos y Departamentos ---")
aisles = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')
print("Duplicados aisles:", aisles['aisle'].duplicated().sum(), "| Espacios:", aisles['aisle'].str.contains(r'^\s|\s$', regex=True).sum())
print("Duplicados departments:", departments['department'].duplicated().sum(), "| Espacios:", departments['department'].str.contains(r'^\s|\s$', regex=True).sum())

print("\n--- 3. Nulos en add_to_cart_order ---")
nulos_cart = op_limpio[op_limpio['add_to_cart_order'].isna()]
tamano_pedidos = op_limpio[op_limpio['order_id'].isin(nulos_cart['order_id'])].groupby('order_id').size()
print("Pedidos afectados:", nulos_cart['order_id'].nunique(), "| Mínimo de productos en estos pedidos:", tamano_pedidos.min())
print("Justificación: Son pedidos gigantes de más de 15 artículos, probablemente un error de registro en carritos muy grandes.")

print("\n--- 4. Nombres cambiados al limpiar ---")
diferentes = (products_crudo['product_name'].notna()) & (products_crudo['product_name'] != products_limpio['product_name'])
print("Nombres estandarizados/cambiados:", diferentes.sum())

print("\n--- 5. Memoria: Texto vs Categoría ---")
mapa = {0:'Dom', 1:'Lun', 2:'Mar', 3:'Mie', 4:'Jue', 5:'Vie', 6:'Sab'}
serie_texto = orders_limpio['order_dow'].map(mapa)
serie_cat = pd.Categorical(serie_texto, categories=list(mapa.values()), ordered=True)
print(f"Memoria Texto: {serie_texto.memory_usage(deep=True)/1e6:.1f} MB | Memoria Categoría: {serie_cat.memory_usage(deep=True)/1e6:.1f} MB")

print("\n--- 6. Productos Unknown en pedidos ---")
unk_ids = products_limpio[products_limpio['product_name'] == 'Unknown']['product_id']
unk_pedidos = op_limpio[op_limpio['product_id'].isin(unk_ids)]
print(f"Aparecen en {len(unk_pedidos)} registros. No se deben borrar porque alteraría el total de artículos y facturación del cliente.")

print("\n--- 7. Reemplazo de missing por pd.NA ---")
dept_prueba = departments.copy()
dept_prueba['department'] = dept_prueba['department'].replace('missing', pd.NA)
print("No conviene usar pd.NA en la original antes de agrupar, porque el groupby ignora los nulos por defecto y perderíamos esa data.")

--- 1. Duplicados en order_prods ---
Exactos: 0
Pares repetidos: 0
Valores en reordered: [0 1]

--- 2. Pasillos y Departamentos ---
Duplicados aisles: 0 | Espacios: 0
Duplicados departments: 0 | Espacios: 0

--- 3. Nulos en add_to_cart_order ---
Pedidos afectados: 70 | Mínimo de productos en estos pedidos: 65
Justificación: Son pedidos gigantes de más de 15 artículos, probablemente un error de registro en carritos muy grandes.

--- 4. Nombres cambiados al limpiar ---
Nombres estandarizados/cambiados: 6321

--- 5. Memoria: Texto vs Categoría ---
Memoria Texto: 24.9 MB | Memoria Categoría: 0.5 MB

--- 6. Productos Unknown en pedidos ---
Aparecen en 9547 registros. No se deben borrar porque alteraría el total de artículos y facturación del cliente.

--- 7. Reemplazo de missing por pd.NA ---
No conviene usar pd.NA en la original antes de agrupar, porque el groupby ignora los nulos por defecto y perderíamos esa data.


In [ ]:
import numpy as np

# --- Laboratorio 3: Perfil de hábitos de compra ---
# 1. Pedidos por cliente y segmentación
pedidos_por_cliente = orders_limpio.groupby('user_id')['order_id'].count()
segmento_cliente = pd.cut(pedidos_por_cliente, 
                          bins=[0, 1, 4, np.inf], 
                          labels=['ocasional', 'regular', 'frecuente'])

# Llevamos el segmento a cada pedido
orders_limpio['segmento'] = orders_limpio['user_id'].map(segmento_cliente)

# 2. Resumen por segmento
resumen_segmento = (orders_limpio
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
print("--- Resumen por Segmento ---")
display(resumen_segmento)

# 3. Franjas horarias y tabla cruzada
orders_limpio['franja'] = pd.cut(orders_limpio['order_hour_of_day'], 
                                 bins=[-1, 5, 11, 17, 23], 
                                 labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders_limpio['segmento'], orders_limpio['franja'], normalize='index')
          .mul(100).round(1))
print("\n--- Perfil Horario por Segmento (%) ---")
display(perfil)

# 4. Limpieza de auxiliares para no arrastrarlas a la Lección 4
orders_limpio = orders_limpio.drop(columns=['segmento', 'franja'])